# 02 - Data Preprocessing and Feature Engineering

## Project: Banking Loan Default Prediction

### Objectives
1. Load and inspect the raw dataset.
2. Separate features and target variable.
3. Remove unnecessary columns.
4. Handle missing values.
5. Perform feature engineering.
6. Encode categorical variables.
7. Scale numerical features where required.
8. Prepare data for model training.

In [1]:
# Import required libraries

import pandas as pd
import numpy as np

from pathlib import Path

# Locate project directory
if (Path.cwd() / "data").exists():
    PROJECT_ROOT = Path.cwd()
elif (Path.cwd().parent / "data").exists():
    PROJECT_ROOT = Path.cwd().parent
else:
    raise FileNotFoundError("Project root not found!")

# Load raw dataset
DATA_PATH = PROJECT_ROOT / "data" / "raw" / "banking_loan_default_dataset.csv"

df = pd.read_csv(DATA_PATH)

print("Dataset loaded successfully!")
print("Dataset shape:", df.shape)

# Display first five records
display(df.head())

Dataset loaded successfully!
Dataset shape: (100000, 29)


,loan_id,customer_id,application_date,age,gender,marital_status,education_level,employment_type,employment_years,annual_income_lkr,...,loan_purpose,loan_amount_lkr,loan_term_months,interest_rate_pct,monthly_emi_lkr,debt_to_income_ratio,has_cosigner,collateral_type,collateral_value_lkr,default_flag
0,LN0000001,CUST0000001,2024-11-25,25,Female,Married,Secondary,Contract,7.0,1931156.07,...,Vehicle,515829.67,60,12.08,11495.21,0.4894,No,No Collateral,0.00,0
1,LN0000002,CUST0000002,2024-11-09,59,Male,Married,PhD,Salaried,17.8,1243811.73,...,Education,2439406.51,12,12.33,217115.08,2.5397,No,Fixed Deposit,2196095.97,1
2,LN0000003,CUST0000003,2022-09-28,53,Female,Married,Bachelor,Salaried,4.7,1395198.42,...,Housing,345290.23,36,9.66,11086.51,0.1507,No,No Collateral,0.00,0
3,LN0000004,CUST0000004,2023-09-15,42,Male,Married,Master,Salaried,2.1,3062216.47,...,Housing,2138803.12,84,10.91,36520.39,0.4927,Yes,No Collateral,0.00,0
4,LN0000005,CUST0000005,2020-01-03,42,Female,Married,Bachelor,Unemployed,13.1,2541211.95,...,Debt Consolidation,2270250.55,48,8.93,56419.85,0.3624,No,No Collateral,0.00,0


In [2]:
# Inspect dataset information

print("Dataset Information:")
df.info()

print("\nTarget Variable Distribution:")
print(df["default_flag"].value_counts())

print("\nMissing Values:")
print(df.isnull().sum()[df.isnull().sum() > 0])

Dataset Information:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100000 entries, 0 to 99999
Data columns (total 29 columns):
 #   Column                   Non-Null Count   Dtype  
---  ------                   --------------   -----  
 0   loan_id                  100000 non-null  object 
 1   customer_id              100000 non-null  object 
 2   application_date         100000 non-null  object 
 3   age                      100000 non-null  int64  
 4   gender                   100000 non-null  object 
 5   marital_status           100000 non-null  object 
 6   education_level          100000 non-null  object 
 7   employment_type          100000 non-null  object 
 8   employment_years         97408 non-null   float64
 9   annual_income_lkr        98433 non-null   float64
 10  monthly_income_lkr       100000 non-null  float64
 11  home_ownership           97971 non-null   object 
 12  existing_loans           100000 non-null  int64  
 13  outstanding_debt_lkr     100000 non-nul

In [3]:
# Step 6.2 - Separate Features and Target

# Separate target variable
X = df.drop("default_flag", axis=1)

# Store target variable
y = df["default_flag"]

# Remove unnecessary identifier columns
identifier_columns = ["loan_id", "customer_id"]

X = X.drop(columns=identifier_columns)

# Display results
print("Features and target separated successfully!")

print("\nFeatures shape:", X.shape)
print("Target shape:", y.shape)

print("\nFeature columns:")
print(X.columns.tolist())

print("\nTarget distribution:")
print(y.value_counts())

Features and target separated successfully!

Features shape: (100000, 26)
Target shape: (100000,)

Feature columns:
['application_date', 'age', 'gender', 'marital_status', 'education_level', 'employment_type', 'employment_years', 'annual_income_lkr', 'monthly_income_lkr', 'home_ownership', 'existing_loans', 'outstanding_debt_lkr', 'credit_score', 'credit_history_years', 'previous_defaults', 'delinquencies_last_2yrs', 'credit_utilization', 'loan_purpose', 'loan_amount_lkr', 'loan_term_months', 'interest_rate_pct', 'monthly_emi_lkr', 'debt_to_income_ratio', 'has_cosigner', 'collateral_type', 'collateral_value_lkr']

Target distribution:
default_flag
0    66151
1    33849
Name: count, dtype: int64


In [4]:
# Step 6.3 - Train-Test Split

from sklearn.model_selection import train_test_split

# Split dataset into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

# Display dataset sizes
print("Train-Test Split Completed Successfully!")

print("\nTraining Features:", X_train.shape)
print("Testing Features:", X_test.shape)

print("\nTraining Target:", y_train.shape)
print("Testing Target:", y_test.shape)

print("\nTraining Target Distribution:")
print(y_train.value_counts())

print("\nTesting Target Distribution:")
print(y_test.value_counts())

Train-Test Split Completed Successfully!

Training Features: (80000, 26)
Testing Features: (20000, 26)

Training Target: (80000,)
Testing Target: (20000,)

Training Target Distribution:
default_flag
0    52921
1    27079
Name: count, dtype: int64

Testing Target Distribution:
default_flag
0    13230
1     6770
Name: count, dtype: int64


In [5]:
# Step 6.4 - Date Feature Engineering

# Convert application_date into datetime format
X_train["application_date"] = pd.to_datetime(
    X_train["application_date"]
)

X_test["application_date"] = pd.to_datetime(
    X_test["application_date"]
)

# Extract date features from training data
for data in [X_train, X_test]:
    data["application_year"] = data["application_date"].dt.year
    data["application_month"] = data["application_date"].dt.month
    data["application_day"] = data["application_date"].dt.day
    data["application_dayofweek"] = data["application_date"].dt.dayofweek

# Remove original application_date column
X_train.drop(columns=["application_date"], inplace=True)
X_test.drop(columns=["application_date"], inplace=True)

# Display results
print("Date Feature Engineering Completed Successfully!")

print("\nNew Features:")
print([
    "application_year",
    "application_month",
    "application_day",
    "application_dayofweek"
])

print("\nTraining Features Shape:", X_train.shape)
print("Testing Features Shape:", X_test.shape)

display(X_train.head())

Date Feature Engineering Completed Successfully!

New Features:
['application_year', 'application_month', 'application_day', 'application_dayofweek']

Training Features Shape: (80000, 29)
Testing Features Shape: (20000, 29)


,age,gender,marital_status,education_level,employment_type,employment_years,annual_income_lkr,monthly_income_lkr,home_ownership,existing_loans,...,interest_rate_pct,monthly_emi_lkr,debt_to_income_ratio,has_cosigner,collateral_type,collateral_value_lkr,application_year,application_month,application_day,application_dayofweek
31154,31,Male,Married,Bachelor,Salaried,4.3,3903471.05,325289.25,Own,2,...,13.35,322846.20,1.4368,No,No Collateral,0.00,2024,2,5,0
42648,24,Female,Married,Diploma,Business Owner,6.0,9102721.46,758560.12,Rent,0,...,13.22,178299.91,0.3919,No,Property,11958377.40,2020,11,23,0
14091,69,Male,Single,Secondary,Business Owner,5.7,742363.12,61863.59,Mortgage,0,...,10.97,34985.24,0.8412,No,Property,1203792.69,2019,5,1,2
33565,21,Female,Married,Diploma,Salaried,3.0,4834665.09,402888.76,Rent,0,...,10.83,429303.89,1.3542,No,Vehicle,5511135.55,2024,5,31,4
28703,58,Male,Single,Secondary,Salaried,11.3,6810479.92,567539.99,Own,0,...,14.03,89485.80,0.3715,No,Property,4775168.17,2020,4,16,3


In [6]:
# Step 6.5 - Handling Missing Values

from sklearn.impute import SimpleImputer

# Identify numerical and categorical columns
numerical_columns = X_train.select_dtypes(
    include=["int64", "float64"]
).columns.tolist()

categorical_columns = X_train.select_dtypes(
    include=["object"]
).columns.tolist()

print("Numerical columns:", len(numerical_columns))
print("Categorical columns:", len(categorical_columns))

# Create imputers
numerical_imputer = SimpleImputer(strategy="median")
categorical_imputer = SimpleImputer(strategy="most_frequent")

# Fit and transform training data
X_train[numerical_columns] = numerical_imputer.fit_transform(
    X_train[numerical_columns]
)

X_train[categorical_columns] = categorical_imputer.fit_transform(
    X_train[categorical_columns]
)

# Transform testing data using training imputers
X_test[numerical_columns] = numerical_imputer.transform(
    X_test[numerical_columns]
)

X_test[categorical_columns] = categorical_imputer.transform(
    X_test[categorical_columns]
)

# Check remaining missing values
print("\nMissing values after imputation:")
print("Training:", X_train.isnull().sum().sum())
print("Testing:", X_test.isnull().sum().sum())

print("\nMissing Value Handling Completed Successfully!")

Numerical columns: 17
Categorical columns: 8

Missing values after imputation:
Training: 0
Testing: 0

Missing Value Handling Completed Successfully!


In [7]:
# Step 6.6 - Categorical Feature Encoding

from sklearn.preprocessing import OneHotEncoder

# Identify categorical columns
categorical_columns = X_train.select_dtypes(
    include=["object", "category"]
).columns.tolist()

# Identify numerical columns
numerical_columns = X_train.select_dtypes(
    include=["int64", "float64"]
).columns.tolist()

print("Categorical columns:", categorical_columns)
print("\nNumber of categorical columns:", len(categorical_columns))
print("Number of numerical columns:", len(numerical_columns))

# Create One-Hot Encoder
try:
    encoder = OneHotEncoder(
        handle_unknown="ignore",
        sparse_output=False
    )
except TypeError:
    encoder = OneHotEncoder(
        handle_unknown="ignore",
        sparse=False
    )

# Fit encoder on training data and transform
X_train_encoded = encoder.fit_transform(
    X_train[categorical_columns]
)

# Transform testing data using the fitted encoder
X_test_encoded = encoder.transform(
    X_test[categorical_columns]
)

# Create DataFrames from encoded features
encoded_columns = encoder.get_feature_names_out(
    categorical_columns
)

X_train_encoded_df = pd.DataFrame(
    X_train_encoded,
    columns=encoded_columns,
    index=X_train.index
)

X_test_encoded_df = pd.DataFrame(
    X_test_encoded,
    columns=encoded_columns,
    index=X_test.index
)

# Combine numerical and encoded categorical features
X_train = pd.concat(
    [X_train[numerical_columns], X_train_encoded_df],
    axis=1
)

X_test = pd.concat(
    [X_test[numerical_columns], X_test_encoded_df],
    axis=1
)

# Display results
print("\nCategorical Encoding Completed Successfully!")

print("\nTraining dataset shape:", X_train.shape)
print("Testing dataset shape:", X_test.shape)

print("\nAll training columns are numerical:",
      X_train.select_dtypes(exclude=["number"]).empty)

print("\nFirst five encoded records:")
display(X_train.head())

Categorical columns: ['gender', 'marital_status', 'education_level', 'employment_type', 'home_ownership', 'loan_purpose', 'has_cosigner', 'collateral_type']

Number of categorical columns: 8
Number of numerical columns: 17

Categorical Encoding Completed Successfully!

Training dataset shape: (80000, 50)
Testing dataset shape: (20000, 50)

All training columns are numerical: True

First five encoded records:


,age,employment_years,annual_income_lkr,monthly_income_lkr,existing_loans,outstanding_debt_lkr,credit_score,credit_history_years,previous_defaults,delinquencies_last_2yrs,...,loan_purpose_Housing,loan_purpose_Medical,loan_purpose_Personal,loan_purpose_Vehicle,has_cosigner_No,has_cosigner_Yes,collateral_type_Fixed Deposit,collateral_type_No Collateral,collateral_type_Property,collateral_type_Vehicle
31154,31.0,4.3,3903471.05,325289.25,2.0,5438256.31,544.0,3.2,0.0,1.0,...,0.0,0.0,1.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0
42648,24.0,6.0,9102721.46,758560.12,0.0,3917980.04,650.0,2.9,0.0,1.0,...,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0
14091,69.0,5.7,742363.12,61863.59,0.0,424512.13,551.0,3.7,0.0,0.0,...,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0
33565,21.0,3.0,4834665.09,402888.76,0.0,2198158.33,628.0,3.0,0.0,0.0,...,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0
28703,58.0,11.3,6810479.92,567539.99,0.0,3516879.20,680.0,4.2,0.0,0.0,...,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0


In [8]:
# Step 6.7 - Feature Scaling

from sklearn.preprocessing import StandardScaler

# Create StandardScaler
scaler = StandardScaler()

# Fit scaler using training data and transform
X_train[numerical_columns] = scaler.fit_transform(
    X_train[numerical_columns]
)

# Transform testing data using the fitted scaler
X_test[numerical_columns] = scaler.transform(
    X_test[numerical_columns]
)

# Display results
print("Feature Scaling Completed Successfully!")

print("\nTraining dataset shape:", X_train.shape)
print("Testing dataset shape:", X_test.shape)

print("\nTraining numerical feature statistics:")
display(X_train[numerical_columns].describe().round(3))

print("\nMissing values:")
print("Training:", X_train.isnull().sum().sum())
print("Testing:", X_test.isnull().sum().sum())

Feature Scaling Completed Successfully!

Training dataset shape: (80000, 50)
Testing dataset shape: (20000, 50)

Training numerical feature statistics:


,age,employment_years,annual_income_lkr,monthly_income_lkr,existing_loans,outstanding_debt_lkr,credit_score,credit_history_years,previous_defaults,delinquencies_last_2yrs,credit_utilization,loan_amount_lkr,loan_term_months,interest_rate_pct,monthly_emi_lkr,debt_to_income_ratio,collateral_value_lkr
count,80000.000,80000.000,80000.000,80000.000,80000.000,80000.000,80000.000,80000.000,80000.000,80000.000,80000.000,80000.000,80000.000,80000.000,80000.000,80000.000,80000.000
mean,-0.000,-0.000,-0.000,0.000,-0.000,0.000,-0.000,-0.000,-0.000,-0.000,-0.000,0.000,-0.000,0.000,0.000,-0.000,0.000
std,1.000,1.000,1.000,1.000,1.000,1.000,1.000,1.000,1.000,1.000,1.000,1.000,1.000,1.000,1.000,1.000,1.000
min,-1.699,-1.593,-1.219,-1.214,-1.095,-0.951,-4.637,-1.618,-0.279,-0.452,-2.047,-1.112,-1.445,-2.353,-0.834,-1.549,-0.660
25%,-0.867,-0.721,-0.652,-0.657,-1.095,-0.624,-0.670,-0.719,-0.279,-0.452,-0.769,-0.707,-0.633,-0.686,-0.586,-0.633,-0.660
50%,0.034,-0.229,-0.261,-0.263,-0.183,-0.304,-0.004,-0.211,-0.279,-0.452,-0.082,-0.297,-0.226,-0.020,-0.314,-0.212,-0.395
75%,0.866,0.486,0.344,0.351,0.729,0.268,0.677,0.504,-0.279,-0.452,0.701,0.374,0.180,0.657,0.194,0.343,0.271
max,1.697,9.252,12.657,12.558,6.200,23.602,2.458,8.253,3.587,10.008,3.119,3.863,2.211,4.349,9.710,5.355,7.080



Missing values:
Training: 0
Testing: 0


In [9]:
# Step 6.8 - Save Processed Datasets

import joblib
from pathlib import Path

# Create output directories
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
MODEL_DIR = PROJECT_ROOT / "artifacts" / "models"

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
MODEL_DIR.mkdir(parents=True, exist_ok=True)

# Save training and testing features
X_train.to_csv(PROCESSED_DIR / "X_train.csv", index=False)
X_test.to_csv(PROCESSED_DIR / "X_test.csv", index=False)

# Save training and testing target variables
y_train.to_csv(PROCESSED_DIR / "y_train.csv", index=False)
y_test.to_csv(PROCESSED_DIR / "y_test.csv", index=False)

# Save preprocessing objects
joblib.dump(numerical_imputer, MODEL_DIR / "numerical_imputer.pkl")
joblib.dump(categorical_imputer, MODEL_DIR / "categorical_imputer.pkl")
joblib.dump(encoder, MODEL_DIR / "onehot_encoder.pkl")
joblib.dump(scaler, MODEL_DIR / "standard_scaler.pkl")

print("Processed datasets saved successfully!")
print("\nPreprocessing objects saved successfully!")

print("\nProcessed dataset files:")
for file in sorted(PROCESSED_DIR.glob("*.csv")):
    print(file.name)

print("\nPreprocessing objects:")
for file in sorted(MODEL_DIR.glob("*.pkl")):
    print(file.name)

Processed datasets saved successfully!

Preprocessing objects saved successfully!

Processed dataset files:
X_test.csv
X_train.csv
y_test.csv
y_train.csv

Preprocessing objects:
categorical_imputer.pkl
numerical_imputer.pkl
onehot_encoder.pkl
standard_scaler.pkl
